# 01 Dwarf and Cluster SIDM Forecast

This notebook reproduces the Phase-1/Phase-2 forecast products for SIDM vs CDM using:
- benchmark halos (`1e10`, `1e14 Msun`),
- local projection (`rho -> Sigma -> DeltaSigma`),
- toy distinguishability metrics and precision sweeps.

All quantities are in physical units unless noted.


## 1. Setup

This cell defines paths and helper imports. Run from repository root with `PYTHONPATH=src` available.


In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd()
outputs_dir = repo_root / "outputs"
figures_dir = outputs_dir / "figures"
tables_dir = outputs_dir / "tables"

print(f"repo_root: {repo_root}")
print(f"outputs_dir: {outputs_dir}")


## 2. Run benchmark products

This executes the three production scripts:
1. benchmark profile and lensing forecasts,
2. NFW reference cross-check,
3. precision sweep (`2sigma/3sigma/5sigma`, full/inner/outer windows).


In [ ]:
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = "src"
run_environment["XDG_CACHE_HOME"] = ".cache"
run_environment["MPLCONFIGDIR"] = ".mplconfig"

commands = [
    ["python", "scripts/run_benchmarks.py", "--sidm-backend", "parametric"],
    ["python", "scripts/run_reference_crosscheck.py"],
    ["python", "scripts/run_precision_sweep.py", "--sidm-backend", "parametric"],
]

for command in commands:
    print("\nRunning:", " ".join(command))
    subprocess.run(command, check=True, cwd=repo_root, env=run_environment)

print("\nAll generation scripts completed.")


## 3. Load key summary tables

`benchmark_summary.csv` gives model-level distinguishability metrics.
`precision_sweep.csv` gives required uniform fractional precision for target significance.
`nfw_reference_crosscheck.csv` gives projection-validation residuals.


In [ ]:
benchmark_summary = pd.read_csv(tables_dir / "benchmark_summary.csv")
precision_sweep = pd.read_csv(tables_dir / "precision_sweep.csv")
reference_crosscheck = pd.read_csv(tables_dir / "nfw_reference_crosscheck.csv")
backend_status = pd.read_csv(tables_dir / "nfw_reference_backend_status.csv")

print("benchmark_summary:")
display(benchmark_summary)

print("precision_sweep (first 20 rows):")
display(precision_sweep.head(20))

print("reference backend status:")
display(backend_status)


## 4. Short interpretation snapshot

This section reports compact, reproducible interpretation numbers directly from generated tables.


In [ ]:
max_ref = reference_crosscheck["frac_diff_local_vs_analytic"].abs().max()
max_ref_percent = 100.0 * max_ref
print(f"Max |local-analytic| NFW DeltaSigma residual: {max_ref_percent:.3f}%")

summary_3sigma_full = (
    precision_sweep
    .query("target_sigma == 3.0 and window == 'full'")
    [[
        "benchmark",
        "sigma_over_m_cm2_g",
        "required_uniform_percent",
        "max_abs_ratio_shift_percent",
    ]]
    .sort_values(["benchmark", "sigma_over_m_cm2_g"])
)

print("\nRequired uniform precision (%) for 3sigma in full window:")
display(summary_3sigma_full)

best_per_benchmark = (
    summary_3sigma_full
    .sort_values(["benchmark", "required_uniform_percent"])
    .groupby("benchmark", as_index=False)
    .first()
)
print("Best (lowest required %) per benchmark for 3sigma/full:")
display(best_per_benchmark)


## 5. Generated figures

The following figures are the main products used in slides and quick-look diagnostics.


In [ ]:
figure_names = [
    "dwarf_rho_profiles.png",
    "cluster_rho_profiles.png",
    "dwarf_delta_sigma_profiles.png",
    "cluster_delta_sigma_profiles.png",
    "nfw_reference_crosscheck.png",
    "dwarf_precision_sweep.png",
    "cluster_precision_sweep.png",
]

for figure_name in figure_names:
    figure_path = figures_dir / figure_name
    print(f"\n{figure_name}")
    display(Image(filename=str(figure_path)))


## 6. Reproducibility checklist

- Verify `outputs/figures/CAPTION.md` matches current figure timestamps and interpretations.
- Verify `outputs/INVENTORY.md` includes current `outputs/tables` and `outputs/intermediate` files.
- Archive this notebook with output state when preparing handoff material.
